# ベースライン + 過去コンペ上位解法で複数提出ファイルを作成

**train_baseline.ipynb** をベースに、atmaCup #15 などの上位解法を **top_solutions.py** で実装し、
このノートブックを実行すると **複数パターンの提出ファイル** が `outputs/submissions/` に保存される。

- **実装**: 重い処理は `top_solutions.py` に集約（OOF TE・比率・NMF・SVD・既知/未知二モデル・シードアベレージング）
- **提出ファイル**: baseline38, v2, v2_ratio, v2_nmf, v2_svd, v2_all, known_unknown, v2_all_seed3 の 8 パターン
- **流れ**: セルを上から順に実行するだけで、検証（CV）なしで **8 種類の提出ファイル** が `outputs/submissions/` に保存される。オプションで「リークなし版」セルを実行すると `time_aware=True` で同じ 8 ファイルが上書きされる。
- **提出の作り方**: デフォルトで **全 train で 1 本学習** して test を予測（`train_baseline.ipynb` と同じ）。CV は時系列で評価用のみ。
- **既知/未知**: **既知映画用・未知映画用でモデルを分けるのは `submission_known_unknown.csv` のみ**。他 7 パターンは 1 本のモデル（またはシード 3 本の平均）で全 test を予測。

In [ ]:
# archive/ から実行しても lib と top_solutions を参照できるようにする（カーネルはプロジェクトルートで起動するか、このセルを最初に実行）
import sys
from pathlib import Path
_root = Path.cwd().parent if not (Path.cwd() / "lib").exists() else Path.cwd()
_archive = Path.cwd() if _root != Path.cwd() else Path.cwd() / "archive"
for p in (str(_root), str(_archive)):
    if p not in sys.path:
        sys.path.insert(0, p)

In [2]:
import os
import random
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from lib import get_baseline_data, BASELINE_FEATURES
from top_solutions import run_all_submission_configs, verify_submissions
try:
    from top_solutions import retry_failed_submission_configs
except ImportError:
    retry_failed_submission_configs = None  # やり直しは top_solutions.py に関数が無いと使えない

OUTPUT_DIR = "outputs"
os.makedirs(os.path.join(OUTPUT_DIR, "submissions"), exist_ok=True)
print("Setup complete.")

Setup complete.


In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## データ読み込み

In [4]:
train, test = get_baseline_data()
y = train["target"].values
FEATURES_BASELINE = list(BASELINE_FEATURES)
print(f"Train: {len(train):,}, Test: {len(test):,}, ベースライン: {len(FEATURES_BASELINE)} 特徴量")

Train: 653,507, Test: 40,716, ベースライン: 38 特徴量


## 提出ファイル作成（検証なし・上から順に実行）

In [6]:
# 検証（CV）は行わず、8 種類の提出ファイルのみ作成（1つエラーしても続行し、後でやり直し可）
summary_df = run_all_submission_configs(
    train,
    test,
    y,
    baseline_features=FEATURES_BASELINE,
    output_dir=OUTPUT_DIR,
    verbose=True,
    skip_cv=True,
    continue_on_error=True,
)

TypeError: run_all_submission_configs() got an unexpected keyword argument 'continue_on_error'

## 作成された提出ファイル一覧

In [ ]:
for _, r in summary_df.iterrows():
    print(r["path"])
print(f"\n→ 計 {len(summary_df)} ファイル")

# 提出ファイルの検証（行数・列名・target 範囲・ID 一致）
verify_df = verify_submissions(test, OUTPUT_DIR)
if verify_df["ok"].all():
    print("\n検証: 全ファイル OK（行数・ID・target 形式問題なし）")
else:
    display(verify_df)
    print("\n上記の ok=False のファイルを確認してください。")

## （オプション）エラーだった設定だけやり直す

上の実行で一部だけエラーになった場合、このセルを実行すると **エラーになった設定だけ** を再実行する。`summary_df` を更新するので、そのあと「作成された提出ファイル一覧」のセルを再実行すれば検証もやり直せる。

In [ ]:
if retry_failed_submission_configs is None:
    print("やり直し機能を使うには top_solutions.py に retry_failed_submission_configs を定義してください。")
else:
    summary_df = retry_failed_submission_configs(
        train, test, y,
        baseline_features=FEATURES_BASELINE,
        output_dir=OUTPUT_DIR,
        summary_df=summary_df,
        verbose=True,
        skip_cv=True,
    )
    failed = summary_df[summary_df["error"].notna()] if "error" in summary_df.columns else pd.DataFrame()
    if len(failed):
        print("まだエラーの設定:", failed["config"].tolist())
    else:
        print("やり直し完了（エラーなし）")

## （オプション）指定した設定だけ作り直す

すでに提出済みのファイルはそのままで、**1つだけ**作り直したいときに使う。下の `configs_to_run` に作り直したい設定名のリストを書く（例: `["v2_all_seed3"]` で v2_all シード3 だけ）。

In [6]:
# 作り直したい設定だけ指定（他は実行しない）
configs_to_run = ["v2_all_seed3"]  # 例: これだけ作り直す

run_all_submission_configs(
    train, test, y,
    baseline_features=FEATURES_BASELINE,
    output_dir=OUTPUT_DIR,
    verbose=True,
    skip_cv=True,
    configs_to_run=configs_to_run,
)

=== v2_all シードアベレージング（3位・57位） ===
→ エラー: lightgbm.sklearn.LGBMClassifier() got multiple values for keyword argument 'random_state'



,config,n_features,cv_auc_mean,cv_auc_std,path,error
0,v2_all_seed3,None,NaN,NaN,,lightgbm.sklearn.LGBMClassifier() got multiple...


## （オプション）リークなし版で再実行

TE・カウントを「各 fold の学習期間のみ」で計算する `time_aware=True` で再実行すると、同じ 8 ファイルがリークなし版で上書きされる。提出を出し終えたあとで実行する想定。

In [ ]:
summary_df_ta = run_all_submission_configs(
    train, test, y,
    baseline_features=FEATURES_BASELINE,
    output_dir=OUTPUT_DIR,
    verbose=True,
    time_aware=True,
)
display(summary_df_ta)